# Loan Term Visualization

In [14]:
%pip install pandas plotly

Note: you may need to restart the kernel to use updated packages.


In [15]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.


In [16]:
import plotly.io as pio
pio.renderers.default = "browser"

In [1]:
"""
Interactive Plotly recreation of the Tableau viz:
"Borrowers with lower scores are paying more interest with a 36-month loan term."

Metric:
    1. Bin credit scores into buckets of 50 (floor): 350, 400, ... 850
    2. Average interest rate per bucket and loan term
    3. Subtract the average of ALL borrowers in that bucket
       -> interest premium vs. the bucket average (0 = average)

Usage:
    pip install pandas plotly
    (in a Jupyter notebook, run:  %pip install pandas plotly  in a cell first)
    python loan_term_premium_plotly.py        # opens the chart in your browser
"""

import pandas as pd
import plotly.graph_objects as go

DATA_PATH = "loan_dataset.csv"
BLUE = "#6A9CC9"
LIGHT_GRAY = "#D3D3D3"
MID_GRAY = "#B5B5B5"
DARK_GRAY = "#666666"
TITLE_COLOR = "#333333"
FONT = "Tableau Book, Segoe UI, Helvetica Neue, Arial, sans-serif"

# ----------------------------------------------------------------------------
# Data prep
# ----------------------------------------------------------------------------
df = pd.read_csv(DATA_PATH)
df["score_bin"] = (df["credit_score"] // 50) * 50

by_term = df.groupby(["score_bin", "loan_term"])["interest_rate"].agg(["mean", "count"])
avg_rate = by_term["mean"].unstack("loan_term")
counts = by_term["count"].unstack("loan_term")
bin_avg = df.groupby("score_bin")["interest_rate"].mean()
premium = avg_rate.sub(bin_avg, axis=0)

x = premium.index.tolist()


def hover(term):
    """Custom hover text with the raw numbers behind each dot."""
    return [
        f"<b>{term}-month loan term</b><br>"
        f"Credit score bucket: {b}–{b + 49}<br>"
        f"Premium vs. bucket avg: {premium.loc[b, term]:+.2f} pts<br>"
        f"Avg rate: {avg_rate.loc[b, term]:.2f}%<br>"
        f"Borrowers: {int(counts.loc[b, term]):,}"
        for b in x
    ]


# ----------------------------------------------------------------------------
# Figure
# ----------------------------------------------------------------------------
fig = go.Figure()

# 60-month first so 36-month renders on top
fig.add_trace(go.Scatter(
    x=x, y=premium[60], mode="markers", name="60-Month Loan Term",
    marker=dict(size=34, color=LIGHT_GRAY),
    hovertext=hover(60), hoverinfo="text",
))
fig.add_trace(go.Scatter(
    x=x, y=premium[36], mode="markers", name="36-Month Loan Term",
    marker=dict(size=34, color=BLUE, opacity=0.9),
    hovertext=hover(36), hoverinfo="text",
))

# Average (zero) line
fig.add_hline(y=0, line=dict(color="#D0D0D0", width=1), layer="below")

# Vertical annotation bar
fig.add_shape(type="line", x0=527, x1=527, y0=0, y1=1.15,
              line=dict(color="#BBBBBB", width=3), layer="below")

# --- Text annotations (HTML spans give us multi-colored text) ----------------
def note(text, x, y, size, color=DARK_GRAY, xref="paper", yref="paper",
         xanchor="left", yanchor="top", **kw):
    fig.add_annotation(
        text=text, x=x, y=y, xref=xref, yref=yref, showarrow=False,
        xanchor=xanchor, yanchor=yanchor, align="left",
        font=dict(size=size, color=color, family=FONT), **kw,
    )


# Headline
note(
    f"Borrowers with lower scores are paying more<br>"
    f"interest with a <span style='color:{BLUE}'>36-month loan term</span>.",
    x=0, y=1.0, size=34, color=TITLE_COLOR,
)

# Sub-headline
note(
    f"Compared to borrowers with a "
    f"<b style='color:{MID_GRAY}'>60-month loan term</b> who "
    f"<b>pay at or below<br>average</b>, those with lower credit scores "
    f"<b style='color:{BLUE}'>pay more in interest premiums</b> with<br>"
    f"the <b style='color:{BLUE}'>36-month loan term</b>.",
    x=0, y=0.865, size=19,
)

# Chart title
note("<b>Borrowers Score By Paid Interest Rates</b>", x=0, y=0.70, size=14)

# Explanatory note next to the vertical bar
note(
    "The <b>Average line</b> represents borrowers paying the standard<br>"
    "interest rates with no added premiums. <b>Points above this<br>"
    "line indicate higher premium payments</b>, while it is better to<br>"
    "stay below this line to keep costs from increasing.",
    x=535, y=1.13, xref="x", yref="y", size=12, xshift=4,
)

# "Average" label on the zero line
note("Average", x=352, y=0.03, xref="x", yref="y", size=12, yanchor="bottom")

# Direct labels (replace the legend)
note(f"<b>60-Month Loan Term</b>", x=872, y=0.30, xref="x", yref="y",
     size=20, color=LIGHT_GRAY, yanchor="middle")
note(f"<b>36-Month Loan Term</b>", x=872, y=-0.14, xref="x", yref="y",
     size=20, color=BLUE, yanchor="middle")

# Source footer
note("Data Source: Loan Predictions 2025, Kaggle | As of 09/2025",
     x=0, y=-0.15, size=12)

# --- Layout ------------------------------------------------------------------
fig.update_layout(
    template="plotly_white",
    font=dict(family=FONT, color=DARK_GRAY),
    showlegend=False,
    width=1100, height=900,
    margin=dict(l=70, r=260, t=20, b=140),
    plot_bgcolor="white", paper_bgcolor="white",
    hoverlabel=dict(bgcolor="white", font=dict(family=FONT, size=13, color=DARK_GRAY)),
    xaxis=dict(
        title=dict(text="Credit Scores", font=dict(size=14)),
        tickmode="array", tickvals=x, tickfont=dict(size=13),
        range=[325, 875], showgrid=False, zeroline=False, ticks="",
    ),
    yaxis=dict(
        title=dict(text="Interest Rates", font=dict(size=14)),
        tickvals=[-1, -0.5, 0, 0.5, 1], tickformat=".1f", tickfont=dict(size=13),
        range=[-1.25, 1.25], showgrid=False, zeroline=False, ticks="",
        domain=[0.0, 0.64],
    ),
)

# Display the chart (inline in a Jupyter notebook, or in your browser from a .py script)
fig.show(config={"displayModeBar": False})